# 🚀 T5 Fine-tuning para Resumización de Textos
## Wikihow y XSum Datasets

Este notebook te permite entrenar un modelo T5 para resumización automática de textos usando Google Colab.

### ¿Qué hace este proyecto?
- **Tarea**: Resumización automática (convertir textos largos en resúmenes cortos)
- **Modelo**: T5-small de Google (60M parámetros)
- **Datasets**: Wikihow (guías) y XSum (noticias BBC)
- **Métrica**: ROUGE-1 (mide calidad del resumen)

### Arquitectura:
```
Texto largo (512 tokens) → T5 Encoder-Decoder → Resumen corto (200 tokens)
```

## 1️⃣ Verificar GPU y Configuración Inicial

In [1]:
# Verificar que tenemos GPU disponible
import torch
print(f"🔧 PyTorch version: {torch.__version__}")
print(f"🎮 CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ No GPU detected! Go to Runtime > Change runtime type > GPU")

🔧 PyTorch version: 2.8.0+cu126
🎮 CUDA available: True
🎮 GPU: Tesla T4
💾 GPU Memory: 15.83 GB


## 2️⃣ Instalar Dependencias

In [2]:
# Instalar librerías necesarias con versión compatible de datasets
!pip install transformers datasets==2.18.0 evaluate rouge-score tensorboard accelerate -q
print("✅ Instalación completada!")
print("📦 Usando datasets v2.18.0 (compatible con scripts y caché de Colab)")

✅ Instalación completada!
📦 Usando datasets v2.18.0 (compatible con scripts y caché de Colab)


## 3️⃣ Montar Google Drive (Opcional - para guardar modelos)

In [3]:
# Descomentar si quieres guardar el modelo en Drive
# from google.colab import drive
# drive.mount('/content/drive')
# SAVE_PATH = '/content/drive/MyDrive/T5_Models'
# !mkdir -p {SAVE_PATH}

# Si no usas Drive, guarda en Colab (se perderá al cerrar sesión)
SAVE_PATH = '/content/T5_Models'
!mkdir -p {SAVE_PATH}
print(f"📁 Modelos se guardarán en: {SAVE_PATH}")

📁 Modelos se guardarán en: /content/T5_Models


## 4️⃣ Importar Librerías y Configuración

In [4]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import T5Tokenizer, T5ForConditionalGeneration
from torch.optim import AdamW  # ✅ Importar AdamW desde PyTorch
from datasets import load_dataset
import evaluate  # ✅ Importar evaluate para métricas
from tqdm.auto import tqdm
import numpy as np
from typing import Dict, List, Tuple
import json
import os

# Configuración del dispositivo
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🔧 Usando dispositivo: {device}")

🔧 Usando dispositivo: cuda


## 5️⃣ Configuración de Hiperparámetros

In [5]:
# ========== CONFIGURACIÓN PRINCIPAL ==========
CONFIG = {
    # Dataset
    'dataset_name': 'xsum',  # Opciones: 'xsum' o 'wikihow'
    'source_max_length': 512,  # Máximo tokens del texto de entrada
    'target_max_length': 200,  # Máximo tokens del resumen

    # Modelo
    'model_name': 't5-small',  # Opciones: 't5-small', 't5-base'

    # Entrenamiento
    'train_batch_size': 22,     # Ajusta según tu GPU (4 para T4, 8 para V100)
    'val_batch_size': 22,
    'num_epochs': 3,           # Aumenta para mejor rendimiento
    'learning_rate': 1e-4,     # Learning rate
    'warmup_steps': 500,       # Pasos de warmup
    'max_grad_norm': 1.0,      # Gradient clipping

    # Evaluación y guardado
    'eval_steps': 500,         # Cada cuántos pasos evaluar
    'save_steps': 1000,        # Cada cuántos pasos guardar checkpoint
    'logging_steps': 50,       # Cada cuántos pasos mostrar loss

    # Generación
    'num_beams': 4,            # Beam search
    'length_penalty': 1.0,
    'repetition_penalty': 2.5,
    'early_stopping': True,

    # Límites de datos (para pruebas rápidas)
    'max_train_samples': None,  # None = usar todo, o un número para limitar
    'max_val_samples': 1000,    # Limitar validación para acelerar
}

print("⚙️ Configuración:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

⚙️ Configuración:
  dataset_name: xsum
  source_max_length: 512
  target_max_length: 200
  model_name: t5-small
  train_batch_size: 22
  val_batch_size: 22
  num_epochs: 3
  learning_rate: 0.0001
  warmup_steps: 500
  max_grad_norm: 1.0
  eval_steps: 500
  save_steps: 1000
  logging_steps: 50
  num_beams: 4
  length_penalty: 1.0
  repetition_penalty: 2.5
  early_stopping: True
  max_train_samples: None
  max_val_samples: 1000


## 6️⃣ Clase Dataset Personalizada

In [6]:
class SummarizationDataset(Dataset):
    """
    Dataset para resumización con T5.
    Soporta XSum y Wikihow.
    """
    def __init__(self, dataset_name: str, split: str, tokenizer,
                 source_max_length: int, target_max_length: int,
                 max_samples: int = None):
        super().__init__()

        # Cargar dataset (datasets v2.18.0 compatible)
        print(f"📚 Cargando dataset {dataset_name} ({split})...")

        if dataset_name == 'xsum':
            # Usar repositorio oficial EdinburghNLP/xsum en formato Parquet
            # Esta es la forma correcta y moderna de cargar el dataset
            self.dataset = load_dataset('EdinburghNLP/xsum', split=split)
            self.source_key = 'document'
            self.target_key = 'summary'
        elif dataset_name == 'wikihow':
            # Cargar Wikihow
            self.dataset = load_dataset("wikihow", "all", split=split)
            self.source_key = 'text'
            self.target_key = 'headline'
        else:
            raise ValueError(f"Dataset no soportado: {dataset_name}. Opciones: 'xsum', 'wikihow'")

        # Limitar muestras si se especifica
        if max_samples and max_samples < len(self.dataset):
            self.dataset = self.dataset.select(range(max_samples))

        self.tokenizer = tokenizer
        self.source_max_length = source_max_length
        self.target_max_length = target_max_length

        print(f"✅ Dataset cargado: {len(self.dataset)} ejemplos")

    def __len__(self) -> int:
        return len(self.dataset)

    def __getitem__(self, idx) -> Dict:
        example = self.dataset[idx]

        # Obtener texto fuente y objetivo
        source_text = example[self.source_key]
        target_text = example[self.target_key]

        # Tokenizar entrada
        source_encoding = self.tokenizer(
            source_text,
            max_length=self.source_max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        # Tokenizar salida
        target_encoding = self.tokenizer(
            target_text,
            max_length=self.target_max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        return {
            'input_ids': source_encoding['input_ids'].squeeze(),
            'attention_mask': source_encoding['attention_mask'].squeeze(),
            'labels': target_encoding['input_ids'].squeeze(),
        }

## 7️⃣ Cargar Datos y Modelo

In [7]:
# Inicializar tokenizador
print(f"🔤 Cargando tokenizador {CONFIG['model_name']}...")
tokenizer = T5Tokenizer.from_pretrained(CONFIG['model_name'])

# Crear datasets
train_dataset = SummarizationDataset(
    dataset_name=CONFIG['dataset_name'],
    split='train',
    tokenizer=tokenizer,
    source_max_length=CONFIG['source_max_length'],
    target_max_length=CONFIG['target_max_length'],
    max_samples=CONFIG['max_train_samples']
)

val_dataset = SummarizationDataset(
    dataset_name=CONFIG['dataset_name'],
    split='validation',
    tokenizer=tokenizer,
    source_max_length=CONFIG['source_max_length'],
    target_max_length=CONFIG['target_max_length'],
    max_samples=CONFIG['max_val_samples']
)

# Crear dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG['train_batch_size'],
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG['val_batch_size'],
    shuffle=False
)

print(f"\n📊 Estadísticas:")
print(f"  Training batches: {len(train_loader)}")
print(f"  Validation batches: {len(val_loader)}")

🔤 Cargando tokenizador t5-small...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


📚 Cargando dataset xsum (train)...
✅ Dataset cargado: 204045 ejemplos
📚 Cargando dataset xsum (validation)...
✅ Dataset cargado: 1000 ejemplos

📊 Estadísticas:
  Training batches: 9275
  Validation batches: 46


In [8]:
# Cargar modelo
print(f"\n🤖 Cargando modelo {CONFIG['model_name']}...")
model = T5ForConditionalGeneration.from_pretrained(CONFIG['model_name'])
model = model.to(device)

# Contar parámetros
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"  Total parámetros: {total_params:,}")
print(f"  Parámetros entrenables: {trainable_params:,}")

# Configurar optimizador
optimizer = AdamW(model.parameters(), lr=CONFIG['learning_rate'])

# Cargar métrica ROUGE usando evaluate
rouge_metric = evaluate.load('rouge')

print("✅ Modelo y optimizador listos!")


🤖 Cargando modelo t5-small...
  Total parámetros: 60,506,624
  Parámetros entrenables: 60,506,624
✅ Modelo y optimizador listos!


## 8️⃣ Funciones de Utilidad

In [9]:
def generate_summaries(model, tokenizer, input_ids, attention_mask, config):
    """
    Genera resúmenes usando beam search.
    """
    model.eval()
    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_length=config['target_max_length'],
            num_beams=config['num_beams'],
            length_penalty=config['length_penalty'],
            repetition_penalty=config['repetition_penalty'],
            early_stopping=config['early_stopping']
        )

    # Decodificar
    summaries = [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]
    return summaries


def evaluate_model(model, tokenizer, val_loader, config, device, max_batches=None):
    """
    Evalúa el modelo en el conjunto de validación.
    """
    model.eval()
    total_loss = 0
    all_predictions = []
    all_references = []

    with torch.no_grad():
        iterator = enumerate(val_loader)
        if max_batches:
            iterator = list(iterator)[:max_batches]

        for i, batch in tqdm(iterator, desc="Evaluando", leave=False):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            # Preparar labels (ignorar padding)
            labels_masked = labels.clone()
            labels_masked[labels_masked == tokenizer.pad_token_id] = -100

            # Calcular loss
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels_masked)
            total_loss += outputs.loss.item()

            # Generar resúmenes para ROUGE (solo algunos batches)
            if i % 10 == 0:  # Evaluar ROUGE solo en 10% de batches (es lento)
                predictions = generate_summaries(model, tokenizer, input_ids, attention_mask, config)
                references = [tokenizer.decode(label, skip_special_tokens=True) for label in labels]
                all_predictions.extend(predictions)
                all_references.extend(references)

    avg_loss = total_loss / len(val_loader if not max_batches else iterator)

    # Calcular ROUGE
    rouge_scores = None
    if all_predictions:
        rouge_scores = rouge_metric.compute(
            predictions=all_predictions,
            references=all_references
        )

    return avg_loss, rouge_scores


def save_checkpoint(model, tokenizer, optimizer, epoch, step, loss, save_path):
    """
    Guarda un checkpoint del modelo.
    """
    checkpoint_dir = os.path.join(save_path, f'checkpoint-epoch{epoch}-step{step}')
    os.makedirs(checkpoint_dir, exist_ok=True)

    # Guardar modelo y tokenizador
    model.save_pretrained(checkpoint_dir)
    tokenizer.save_pretrained(checkpoint_dir)

    # Guardar estado del optimizador
    torch.save({
        'epoch': epoch,
        'step': step,
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': loss,
    }, os.path.join(checkpoint_dir, 'training_state.pt'))

    print(f"💾 Checkpoint guardado en: {checkpoint_dir}")

print("✅ Funciones de utilidad definidas")

✅ Funciones de utilidad definidas


## 9️⃣ Loop de Entrenamiento

In [10]:
def train_model(model, tokenizer, train_loader, val_loader, optimizer, config, device, save_path):
    """
    Loop de entrenamiento principal.
    """
    print("\n🚀 Iniciando entrenamiento...\n")

    global_step = 0
    best_val_loss = float('inf')
    training_history = {'train_loss': [], 'val_loss': [], 'rouge_scores': []}

    for epoch in range(config['num_epochs']):
        print(f"\n{'='*60}")
        print(f"📅 Época {epoch + 1}/{config['num_epochs']}")
        print(f"{'='*60}\n")

        model.train()
        epoch_loss = 0
        progress_bar = tqdm(train_loader, desc=f"Época {epoch+1}")

        for batch_idx, batch in enumerate(progress_bar):
            # Mover datos a GPU
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            # Preparar labels (reemplazar padding con -100)
            labels[labels == tokenizer.pad_token_id] = -100

            # Forward pass
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            loss = outputs.loss
            epoch_loss += loss.item()

            # Backward pass
            optimizer.zero_grad()
            loss.backward()

            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), config['max_grad_norm'])

            optimizer.step()
            global_step += 1

            # Logging
            if global_step % config['logging_steps'] == 0:
                avg_loss = epoch_loss / (batch_idx + 1)
                progress_bar.set_postfix({'loss': f'{avg_loss:.4f}'})
                training_history['train_loss'].append(avg_loss)

            # Evaluación
            if global_step % config['eval_steps'] == 0:
                print(f"\n📊 Evaluando en paso {global_step}...")
                val_loss, rouge_scores = evaluate_model(
                    model, tokenizer, val_loader, config, device, max_batches=50
                )
                training_history['val_loss'].append(val_loss)
                training_history['rouge_scores'].append(rouge_scores)

                print(f"  Validation Loss: {val_loss:.4f}")
                if rouge_scores:
                    # ✅ Formato correcto para evaluate library (devuelve floats directos)
                    print(f"  ROUGE-1: {rouge_scores['rouge1']:.4f}")
                    print(f"  ROUGE-2: {rouge_scores['rouge2']:.4f}")
                    print(f"  ROUGE-L: {rouge_scores['rougeL']:.4f}")

                # Guardar mejor modelo
                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    best_model_dir = os.path.join(save_path, 'best_model')
                    model.save_pretrained(best_model_dir)
                    tokenizer.save_pretrained(best_model_dir)
                    print(f"  🏆 ¡Nuevo mejor modelo guardado! (loss: {val_loss:.4f})")

                model.train()

            # Guardar checkpoint
            if global_step % config['save_steps'] == 0:
                save_checkpoint(model, tokenizer, optimizer, epoch, global_step, loss.item(), save_path)

        # Resumen de época
        avg_epoch_loss = epoch_loss / len(train_loader)
        print(f"\n📈 Época {epoch+1} completada - Loss promedio: {avg_epoch_loss:.4f}")

    # Guardar modelo final
    final_model_dir = os.path.join(save_path, 'final_model')
    model.save_pretrained(final_model_dir)
    tokenizer.save_pretrained(final_model_dir)
    print(f"\n✅ Entrenamiento completado! Modelo final guardado en: {final_model_dir}")

    # Guardar historial
    with open(os.path.join(save_path, 'training_history.json'), 'w') as f:
        # Convertir rouge scores a formato serializable
        history_to_save = {
            'train_loss': training_history['train_loss'],
            'val_loss': training_history['val_loss'],
        }
        json.dump(history_to_save, f, indent=2)

    return training_history

## 🔟 ¡ENTRENAR EL MODELO!

In [ ]:
# ⚠️ EJECUTA ESTA CELDA PARA INICIAR EL ENTRENAMIENTO
history = train_model(
    model=model,
    tokenizer=tokenizer,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    config=CONFIG,
    device=device,
    save_path=SAVE_PATH
)


🚀 Iniciando entrenamiento...


📅 Época 1/3



Época 1:   0%|          | 0/9275 [00:00<?, ?it/s]


📊 Evaluando en paso 500...


Evaluando:   0%|          | 0/46 [00:00<?, ?it/s]

  Validation Loss: 2.5809
  ROUGE-1: 0.2806
  ROUGE-2: 0.0758
  ROUGE-L: 0.2184
  🏆 ¡Nuevo mejor modelo guardado! (loss: 2.5809)

📊 Evaluando en paso 1000...


Evaluando:   0%|          | 0/46 [00:00<?, ?it/s]

## 1️⃣1️⃣ Visualizar Resultados

In [ ]:
import matplotlib.pyplot as plt

# Graficar pérdidas
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history['train_loss'], label='Train Loss')
plt.xlabel('Steps (x logging_steps)')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history['val_loss'], label='Validation Loss', color='orange')
plt.xlabel('Evaluations')
plt.ylabel('Loss')
plt.title('Validation Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_PATH, 'training_curves.png'), dpi=150)
plt.show()

print(f"📊 Gráficas guardadas en: {os.path.join(SAVE_PATH, 'training_curves.png')}")

## 1️⃣2️⃣ Probar el Modelo - Generar Resúmenes

In [ ]:
def test_summarization(text: str, model, tokenizer, device, max_length=512):
    """
    Genera un resumen para un texto dado.
    """
    model.eval()

    # Tokenizar
    inputs = tokenizer(
        text,
        max_length=max_length,
        truncation=True,
        return_tensors='pt'
    )
    input_ids = inputs['input_ids'].to(device)
    attention_mask = inputs['attention_mask'].to(device)

    # Generar
    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_length=200,
            num_beams=4,
            length_penalty=1.0,
            repetition_penalty=2.5,
            early_stopping=True
        )

    # Decodificar
    summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return summary


# Ejemplo de uso
test_text = """
The Amazon rainforest is a moist broadleaf tropical rainforest in the Amazon biome
that covers most of the Amazon basin of South America. This basin encompasses 7 million
square kilometers, of which 5.5 million square kilometers are covered by the rainforest.
This region includes territory belonging to nine nations and 3,344 formally acknowledged
indigenous territories. The majority of the forest is contained within Brazil, with 60%
of the rainforest, followed by Peru with 13%, Colombia with 10%, and with minor amounts
in Bolivia, Ecuador, French Guiana, Guyana, Suriname, and Venezuela.
"""

print("📝 Texto original:")
print(test_text)
print("\n" + "="*60 + "\n")

summary = test_summarization(test_text, model, tokenizer, device)
print("📋 Resumen generado:")
print(summary)

## 1️⃣3️⃣ Evaluar en Ejemplos del Dataset

In [ ]:
# Probar con ejemplos reales del dataset de validación
model.eval()
num_examples = 3

for i in range(num_examples):
    example = val_dataset[i]

    # Obtener texto original y referencia
    input_ids = example['input_ids'].unsqueeze(0).to(device)
    attention_mask = example['attention_mask'].unsqueeze(0).to(device)
    labels = example['labels']

    # Decodificar textos
    original_text = tokenizer.decode(input_ids[0], skip_special_tokens=True)
    reference_summary = tokenizer.decode(labels, skip_special_tokens=True)

    # Generar resumen
    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_length=200,
            num_beams=4,
            repetition_penalty=2.5,
            early_stopping=True
        )
    generated_summary = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Mostrar resultados
    print(f"\n{'='*80}")
    print(f"📄 EJEMPLO {i+1}")
    print(f"{'='*80}\n")
    print(f"📝 Texto original ({len(original_text)} caracteres):")
    print(f"{original_text[:300]}..." if len(original_text) > 300 else original_text)
    print(f"\n✅ Resumen de referencia:")
    print(f"{reference_summary}")
    print(f"\n🤖 Resumen generado:")
    print(f"{generated_summary}")
    print()

## 1️⃣4️⃣ Cargar Modelo Guardado (Para Continuar Entrenamiento)

In [ ]:
# Si quieres cargar un modelo previamente guardado
# Descomenta y ajusta la ruta:

# from transformers import T5ForConditionalGeneration, T5Tokenizer

# checkpoint_path = os.path.join(SAVE_PATH, 'best_model')  # o 'final_model'
# loaded_model = T5ForConditionalGeneration.from_pretrained(checkpoint_path)
# loaded_tokenizer = T5Tokenizer.from_pretrained(checkpoint_path)
# loaded_model = loaded_model.to(device)

# print(f"✅ Modelo cargado desde: {checkpoint_path}")

## 1️⃣5️⃣ Exportar Modelo para Uso Futuro

In [ ]:
# Crear un archivo con instrucciones de uso
usage_instructions = f"""
# Cómo usar este modelo

## Cargar el modelo:

```python
from transformers import T5ForConditionalGeneration, T5Tokenizer

model = T5ForConditionalGeneration.from_pretrained('{SAVE_PATH}/best_model')
tokenizer = T5Tokenizer.from_pretrained('{SAVE_PATH}/best_model')
```

## Generar un resumen:

```python
text = "Tu texto largo aquí..."
inputs = tokenizer(text, max_length=512, truncation=True, return_tensors='pt')
outputs = model.generate(**inputs, max_length=200, num_beams=4)
summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(summary)
```

## Configuración del modelo:
- Dataset: {CONFIG['dataset_name']}
- Modelo base: {CONFIG['model_name']}
- Épocas entrenadas: {CONFIG['num_epochs']}
- Learning rate: {CONFIG['learning_rate']}
"""

with open(os.path.join(SAVE_PATH, 'USAGE.md'), 'w', encoding='utf-8') as f:
    f.write(usage_instructions)

print(f"📝 Instrucciones de uso guardadas en: {os.path.join(SAVE_PATH, 'USAGE.md')}")
print("\n✅ ¡Todo listo! Tu modelo está entrenado y guardado.")